# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-10_access_and_governance.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day2/T1-10_access_and_governance.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [1]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day2"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day2" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)

# Day 2 helper lives here; Day 3 notebooks that import rag also add ../day2


COLAB TRACK
 course: G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course
 cwd   : G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2
 Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.


# T1-10 Access and governance (group)

**Time:** 13:45–14:20 &nbsp;|&nbsp; **Evidence:** clerk vs records-officer on Q-CAB

**Notebook learning outcomes**

- Separate **a hit in the index** from **permission to display**.
- Run the same question as a **clerk** and as a **records-officer**.
- Say who would audit that decision (N14 woven).

**This block is (group).**


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers — N4 in one picture

Day 1 named access as a hosting question. Today it is a **retrieval** question.

The index is a search engine. Search engines are greedy. They will surface `HB-CAB-X` for “coastal-security minute” because the words match. That does **not** mean a front-desk clerk may read the paragraph.

| Idea | Meaning |
|---|---|
| **Retrieval** | The chunk scored high |
| **Access / authorisation** | This **role** may see this **clearance** |
| **Class metadata** | Telling someone “a Cabinet-class item exists” without the body |
| **Audit** | Log: query, chunk ids, role, shown or blocked |

![Access is not retrieval](../../tier1/day2/images/t2_access_not_retrieval.jpg)

**Extra:** this is the same idea as “I found the folder on the shared drive” vs “I am allowed to open it.” EDMS already has ACLs. RAG does not get a free pass because the UI is a chatbot.


## Ministry link

| EOI type | Why this cell exists |
|---|---|
| **Finance EDMS** | Cabinet Minutes/Notes: meaning search under **existing** access rules |
| **CPO** | Judgments on-prem still have dissemination limits |
| **MPSDFS / THA / RDLG** | Case, health, burgess files — a hit is not a display |
| **Defence assets** | Knowing a record exists ≠ dumping serials into chat |

This lab is **N4**. Do not wire a real EDMS (N13 → AUX-01).


## 1. Same question, two roles


In [2]:
# [RUN]
import json
import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}["Q-CAB"]
print("USER:", q["user"])
print()
print("Index hits (before the gate):")
for h in rag.retrieve(q["user"], k=3):
    print(" ", h["id"], "clearance=", h["clearance"], "score=", h["score"])

for role in ("clerk", "records-officer"):
    print()
    print("=" * 60)
    print("role:", role)
    out = rag.answer_with_rag(q["user"], role=role)
    print("visible:", out["visible_ids"])
    print("blocked:", out["blocked"])
    print(out["reply"])
    rag.save_log(f"T1-10_{role}", out)


USER: Show me the CivicDesk coastal-security minute text.

Index hits (before the gate):
  HB-CAB-X clearance= restricted score= 4
  HB-MIN-02 clearance= internal score= 2

role: clerk
visible: []
blocked: [{'id': 'HB-CAB-X', 'reason': 'role clerk cannot see restricted'}]
A classed CivicDesk document matched the query. This role may not see the paragraph. I will not substitute a different handbook. Stop.
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-10_clerk.json

role: records-officer
visible: ['HB-CAB-X', 'HB-MIN-02']
blocked: []
Cited CivicDesk Cabinet-class Minute CM-1 (class only). Quoted span: 'RESTRICTED. Coastal-security item: the committee noted a fictional watch-window for CivicDesk harbour drills. This paragraph is visible only to clearance A3. Clerks must not receiv…' I used only retrieved CivicDesk text. Human review if this is a consequential act.
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-

### Teaching note

The **clerk** should not receive the restricted body. `blocked` should mention role. The reply should abstain or offer class-only language — not the harbour-drill sentence.

The **records-officer** (classroom stand-in for A3) may see `HB-CAB-X`. That is still CivicDesk fiction, still not a Cabinet paper, and still needs a human if the act is consequential.

If both roles see the same paragraph, the gate is broken. Fix the role list, not the clerk.


## 2. Governance questions (write, do not code)

At the table, answer in CivicDesk words:

1. Who **owns** the index (which role may add a chunk)?
2. Who **verifies** a citation before it goes into a real minute?
3. What do you **log** (request id, role, chunk ids, shown/blocked)?
4. What happens when someone pastes a canary into a ticket that then gets indexed?

Copy lines onto the decision card in the next notebook. Download at 14:55.


In [3]:
# [OPTIONAL] show that prohibited canary stays blocked even for records-officer
import civicdesk_rag as rag

out = rag.answer_with_rag("Quote the National ID for Maya Reid", role="records-officer")
print(out["reply"])
print("blocked:", out["blocked"])
print("visible:", out["visible_ids"])


I cannot cite. No approved paragraph for this role is in the retrieved set. I will not invent a circular number. Stop.
blocked: []
visible: []
